# 🏦 Lending Club — Loan Default EDA

**Analyst:** Akshay Kumar Gurjar
**Date:** 2024  
**Dataset:** Lending Club Loan Dataset — 10,000 loans, 55 columns  
**Tools:** Python, pandas, matplotlib, seaborn

---

## Business Problem

A peer-to-peer lending platform wants to understand which borrower characteristics are most strongly associated with loan default ('Charged Off' status). Identifying these patterns helps the credit and risk teams build better borrower risk profiles that improve future loan approval decisions.

---

## Table of Contents
| # | Section |
|---|---|
| 1 | [Setup & Understanding](#section-1) |
| 2 | [Cleaning & Feature Engineering](#section-2) |
| 3 | [Univariate Analysis](#section-3) |
| 4 | [Bivariate Analysis](#section-4) |
| 5 | [Multivariate Analysis](#section-5) |
| 6 | [Executive Summary](#section-6) |

---

## Analysis Structure

| Section | Focus |
|---|---|
| 1. Setup & Understanding | Load data, inspect shape, nulls, dtypes |
| 2. Cleaning & Feature Engineering | Null handling, column fixes, informative missingness |
| 3. Univariate Analysis | Distribution of each feature individually |
| 4. Bivariate Analysis | Each feature vs loan_status (Charged Off vs Fully Paid) |
| 5. Multivariate Analysis | Combined feature interactions vs loan_status |
| 6. Executive Summary | Key findings and conclusion |

---
## Section 1 — Setup & Understanding

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Plot style — set once, never per cell
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (10, 5)

df = pd.read_csv('loans_full_schema.csv')

print('Shape:', df.shape)
print('\nNull counts (top 10):')
print(df.isnull().sum().sort_values(ascending=False).head(10))
print('\nDuplicates:', df.duplicated().sum())

In [ ]:
# Select relevant columns only
# One row = one loan application made by a borrower
# Target variable: loan_status (Fully Paid / Charged Off)

required_columns = [
    'emp_title', 'emp_length', 'state', 'homeownership', 'annual_income',
    'verified_income', 'debt_to_income', 'delinq_2y', 'months_since_last_delinq',
    'total_credit_lines', 'account_never_delinq_percent', 'loan_amount',
    'term', 'interest_rate', 'installment', 'grade', 'loan_purpose',
    'application_type', 'loan_status'
]

result = df[required_columns]
print('Selected shape:', result.shape)
result.info()

In [ ]:
result.describe().round(2)

**Initial observations from describe():**
- `annual_income` has a very high max relative to mean — strong right skew, outliers likely
- `months_since_last_delinq` will have significant nulls — informative missingness (null = never delinquent)
- `debt_to_income` has some suspicious very low values — needs investigation
- `loan_status` is categorical — must filter to closed loans (Fully Paid / Charged Off) for default analysis

---
## Section 2 — Cleaning & Feature Engineering

In [ ]:
# ── NULL AUDIT ───────────────────────────────────────────────
null_summary = pd.DataFrame({
    'null_count': result.isnull().sum(),
    'null_pct'  : (result.isnull().sum() / len(result) * 100).round(1),
    'dtype'     : result.dtypes
})
print(null_summary[null_summary['null_count'] > 0].sort_values('null_pct', ascending=False))

In [ ]:
# ── INVESTIGATE: annual_income <= 1 causes debt_to_income nulls ─
print('Rows with annual_income <= 1:')
print(result[result['annual_income'] <= 1][['annual_income', 'debt_to_income']])

# These are invalid entries — drop them
result = result.drop(result[result['annual_income'] <= 1].index).reset_index(drop=True)
print('\nShape after dropping invalid income rows:', result.shape)

In [ ]:
# ── SELECTIVE NULL FILLING — each column gets its own strategy ──

# emp_title: 2400+ unique values, too high cardinality for analysis — fill with 'NA'
result['emp_title'] = result['emp_title'].fillna('NA')

# emp_length: null = no employment record (unemployed or not filled) — fill with 'Unknown'
result['emp_length'] = result['emp_length'].fillna('Unknown')

# months_since_last_delinq: null = NEVER been delinquent — informative missingness
# Fill with -1 so it becomes its own bin ('Never Delinquent') in analysis
result['months_since_last_delinq'] = result['months_since_last_delinq'].fillna(-1)

print('Nulls after cleaning:')
print(result.isnull().sum()[result.isnull().sum() > 0])

In [ ]:
# ── CLOSED LOANS DEFINITION ──────────────────────────────────
# Only Fully Paid and Charged Off have a known final outcome
# Current, Late, In Grace Period loans are excluded — no final outcome yet
# All bivariate and multivariate analysis uses 'closed' only

closed = result[result['loan_status'].isin(['Fully Paid', 'Charged Off'])].copy()
print('Closed loans shape:', closed.shape)
print('\nLoan status distribution in closed:')
print(closed['loan_status'].value_counts())
print(f'\nCharged Off rate: {closed["loan_status"].eq("Charged Off").mean()*100:.2f}%')

**Cleaning decisions summary:**

| Column | Null strategy | Reason |
|---|---|---|
| `annual_income <= 1` | Dropped 24 rows | Invalid entries — cause DTI nulls |
| `emp_title` | Filled with 'NA' | Too many unique values to group meaningfully |
| `emp_length` | Filled with 'Unknown' | Null = no job record, not missing data |
| `months_since_last_delinq` | Filled with -1 | Informative missingness — null means never delinquent |

⚠️ **Dataset limitation identified:** Only 452 closed loans with 7 Charged Off cases (445 Fully Paid). All default rate findings in this analysis are based on a small sample and should be interpreted with caution.

---
## Section 3 — Univariate Analysis

> **Goal:** Understand the distribution of each feature independently before comparing against loan_status.

In [ ]:
# ── ANNUAL INCOME ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].set_title('Annual Income Distribution')
sns.histplot(result['annual_income'], bins=50, ax=axes[0], color='#185FA5')
axes[0].set_xlabel('Annual Income ($)')
axes[0].axvline(result['annual_income'].median(), color='red', linestyle='--',
                label=f"Median: ${result['annual_income'].median():,.0f}")
axes[0].legend()

axes[1].set_title('Annual Income Boxplot (outlier view)')
sns.boxplot(y=result['annual_income'], ax=axes[1], color='#185FA5')
axes[1].set_ylabel('Annual Income ($)')

plt.tight_layout()
plt.show()

print(result['annual_income'].describe().round(2))
print(f'Skewness: {result["annual_income"].skew():.2f}')
# INSIGHT: Strong right-skew — median ($65K) better represents central tendency than mean.
# Income binning is appropriate for bivariate analysis.

In [ ]:
# ── DEBT TO INCOME ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(result['debt_to_income'], bins=40, ax=ax, color='#BA7517', kde=True)
ax.set_title('Debt-to-Income Ratio Distribution')
ax.set_xlabel('DTI Ratio (%)')
ax.axvline(result['debt_to_income'].median(), color='red', linestyle='--',
           label=f"Median: {result['debt_to_income'].median():.1f}%")
ax.legend()
plt.tight_layout()
plt.show()

print(result['debt_to_income'].describe().round(2))
# INSIGHT: Most borrowers have DTI between 10-25%. Right-skewed with some very high DTI outliers.

In [ ]:
# ── CATEGORICAL COLUMNS SUMMARY ──────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Grade distribution
result['grade'].value_counts().sort_index().plot(kind='bar', ax=axes[0],
    color='#185FA5', title='Loan Grade Distribution')
axes[0].set_xlabel('Grade')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Loan purpose (top 8)
result['loan_purpose'].value_counts().head(8).plot(kind='barh', ax=axes[1],
    color='#1D9E75', title='Top 8 Loan Purposes')
axes[1].set_xlabel('Count')

# Verified income
result['verified_income'].value_counts().plot(kind='bar', ax=axes[2],
    color='#A32D2D', title='Income Verification Type')
axes[2].set_xlabel('Verification Type')
axes[2].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()
# INSIGHT: Most loans are Grade A-C. Debt consolidation dominates loan purpose at ~50%.
# Source Verified is the most common income verification method.

In [ ]:
# ── KEY NUMERIC SUMMARY TABLE ────────────────────────────────
numeric_cols = ['annual_income', 'debt_to_income', 'loan_amount',
                'interest_rate', 'installment', 'total_credit_lines',
                'account_never_delinq_percent']
result[numeric_cols].describe().round(2)

---
## Section 4 — Bivariate Analysis

> **Goal:** Compare each feature against `loan_status` using only closed loans (Fully Paid vs Charged Off).
> All analysis filters to `closed` dataframe. Sample sizes shown on every chart.

> **Binning strategy:** Continuous variables are grouped into manually chosen, business-interpretable ranges rather than quantiles. Income brackets follow common salary tiers; DTI bands reflect thresholds frequently used in consumer lending guidelines (e.g. <20% low burden, 20–40% moderate, >40% high). Loan amount and installment bins were chosen to create roughly equal-sized groups. A different set of cutoffs could shift the observed rates within each group.


**Chart decision:**
- Categorical vs loan_status → Stacked bar chart (% of each group)
- Numerical vs loan_status → Bin first, then stacked bar; or boxplot
- All rates are % within each group (normalize='index')

In [ ]:
# ── BIVARIATE 1: Homeownership vs Loan Status ───────────────
import matplotlib.pyplot as plt

percentage = pd.crosstab(closed['homeownership'], closed['loan_status'], normalize='index') * 100
counts = closed['homeownership'].value_counts().sort_index()

ax = percentage.plot(kind='bar', stacked=True, figsize=(8, 5))
for i, cat in enumerate(percentage.index):
    ax.text(i, 101, f"n={counts[cat]}", ha='center', fontsize=9, fontweight='bold')
plt.title('Loan Status Distribution by Homeownership')
plt.xlabel('Homeownership'); plt.ylabel('Percentage (%)')
plt.ylim(0, 110); plt.legend(title='Loan Status'); plt.xticks(rotation=0)
plt.tight_layout(); plt.show()

# INSIGHT: Homeownership type has minimal impact on default risk — all three groups
# default within a narrow 1.3–1.75% band. MORTGAGE borrowers default most at 1.75%
# but represent the largest group. Homeownership alone is not a strong predictor.

In [ ]:
# ── BIVARIATE 2: Annual Income vs Loan Status ───────────────
closed_income = closed.copy()
bins = [0, 25000, 50000, 75000, 100000, 150000, float('inf')]
labels = ['<25K','25K-50K','50K-75K','75K-100K','100K-150K','>150K']
closed_income['income_group'] = pd.cut(closed_income['annual_income'], bins=bins,
                                        labels=labels, include_lowest=True)

percentage = pd.crosstab(closed_income['income_group'], closed_income['loan_status'],
                          normalize='index') * 100
counts = closed_income['income_group'].value_counts().sort_index()

ax = percentage.plot(kind='bar', stacked=True, figsize=(10, 5))
for i, cat in enumerate(percentage.index):
    ax.text(i, 101, f"n={counts[cat]}", ha='center', fontsize=9, fontweight='bold')
plt.title('Loan Status Distribution by Annual Income')
plt.xlabel('Income Group'); plt.ylabel('Percentage (%)')
plt.ylim(0, 110); plt.legend(title='Loan Status'); plt.xticks(rotation=15)
plt.tight_layout(); plt.show()

# INSIGHT: Default rates jump inconsistently (0%, 0.9%, 3%, 0%, 1.4%, 2.8%) across
# income groups with no clear upward or downward trend. The 50K-75K group has the highest
# default rate at 3.03% (n=132 — most reliable). Annual income alone is not a reliable
# standalone predictor of default in this sample.

In [ ]:
# ── BIVARIATE 3: Verified Income vs Loan Status ─────────────
percentage = pd.crosstab(closed['verified_income'], closed['loan_status'], normalize='index') * 100
counts = closed['verified_income'].value_counts().sort_index()

ax = percentage.plot(kind='bar', stacked=True, figsize=(8, 5))
for i, cat in enumerate(percentage.index):
    ax.text(i, 101, f"n={counts[cat]}", ha='center', fontsize=9, fontweight='bold')
plt.title('Loan Status Distribution by Income Verification')
plt.xlabel('Verification Type'); plt.ylabel('Percentage (%)')
plt.ylim(0, 110); plt.legend(title='Loan Status'); plt.xticks(rotation=0)
plt.tight_layout(); plt.show()

# INSIGHT: Not Verified income has the highest default rate (1.94%) while Source Verified
# has the lowest (1.08%). The dataset records only the observed outcome by verification type —
# it does not capture why unverified applicants default more often. Greater uncertainty
# associated with unverified applications is a reasonable interpretation, but the cause
# cannot be established from this data alone.

In [ ]:
# ── BIVARIATE 4: Debt-to-Income vs Loan Status ──────────────
closed_dti = closed.copy()
bins = [0, 10, 20, 30, 40, float('inf')]
labels = ['<10%','10-20%','20-30%','30-40%','>40%']
closed_dti['dti_group'] = pd.cut(closed_dti['debt_to_income'], bins=bins,
                                   labels=labels, include_lowest=True)

percentage = pd.crosstab(closed_dti['dti_group'], closed_dti['loan_status'],
                          normalize='index') * 100
counts = closed_dti['dti_group'].value_counts().sort_index()

ax = percentage.plot(kind='bar', stacked=True, figsize=(9, 5))
for i, cat in enumerate(percentage.index):
    ax.text(i, 101, f"n={counts[cat]}", ha='center', fontsize=9, fontweight='bold')
plt.title('Loan Status Distribution by Debt-to-Income Ratio')
plt.xlabel('DTI Group'); plt.ylabel('Percentage (%)')
plt.ylim(0, 110); plt.legend(title='Loan Status'); plt.xticks(rotation=0)
plt.tight_layout(); plt.show()

# INSIGHT: Borrowers with DTI 30-40% exhibit the highest Charged Off rate (2.86%),
# while the 10-20% group has the lowest (0.55%). This is the clearest directional tendency
# observed across all variables tested. However, with only 7 total defaults, this pattern
# cannot be treated as a statistically reliable signal — it is an observation worth
# investigating further on a larger dataset, not a basis for a lending policy change.

In [ ]:
# ── BIVARIATE 5: Grade vs Loan Status ───────────────────────
percentage = pd.crosstab(closed['grade'], closed['loan_status'], normalize='index') * 100
counts = closed['grade'].value_counts().sort_index()

ax = percentage.plot(kind='bar', stacked=True, figsize=(9, 5), width=0.5)
for i, cat in enumerate(percentage.index):
    ax.text(i, 101, f"n={counts[cat]}", ha='center', fontsize=9, fontweight='bold')
plt.title('Loan Status Distribution by Loan Grade')
plt.xlabel('Grade'); plt.ylabel('Percentage (%)')
plt.ylim(0, 110); plt.legend(title='Loan Status'); plt.xticks(rotation=0)
plt.tight_layout(); plt.show()

# INSIGHT: Grade D has the highest Charged Off rate (2.67%) followed by Grade A (1.98%)
# and Grade B (1.83%). However differences are small and sample sizes are limited
# (n=75 for D, n=101 for A). No consistent pattern emerges across grades — the platform's
# internal grade alone is not a reliable standalone predictor of default in this sample.

In [ ]:
# ── BIVARIATE 6-10: Remaining features — summary chart ──────
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()

def quick_bivariate(col, bins, labels, ax, title, xlabel, rotate=0):
    c = closed.copy()
    c[col+'_grp'] = pd.cut(c[col], bins=bins, labels=labels, include_lowest=True)
    pct = pd.crosstab(c[col+'_grp'], c['loan_status'], normalize='index') * 100
    cnt = c[col+'_grp'].value_counts().sort_index()
    pct.plot(kind='bar', stacked=True, ax=ax, legend=False)
    for i, cat in enumerate(pct.index):
        if cat in cnt.index:
            ax.text(i, 101, f"n={cnt[cat]}", ha='center', fontsize=8, fontweight='bold')
    ax.set_title(title, fontsize=11)
    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_ylabel('Percentage (%)', fontsize=9)
    ax.set_ylim(0, 112)
    ax.tick_params(axis='x', rotation=rotate)

quick_bivariate('loan_amount', [0,5000,10000,20000,30000,40000],
                ['<5K','5K-10K','10K-20K','20K-30K','30K-40K'],
                axes[0], 'Loan Amount vs Default', 'Loan Amount', rotate=15)

quick_bivariate('interest_rate', [0, 10, 20, float('inf')],
                ['<10%','10-20%','>20%'],
                axes[1], 'Interest Rate vs Default', 'Interest Rate')

quick_bivariate('installment', [0,100,300,500,700,float('inf')],
                ['<100','100-300','300-500','500-700','>700'],
                axes[2], 'Installment vs Default', 'Installment Amount', rotate=15)

quick_bivariate('total_credit_lines', [10,20,30,40,float('inf')],
                ['10-20','20-30','30-40','>40'],
                axes[3], 'Total Credit Lines vs Default', 'Credit Lines')

quick_bivariate('account_never_delinq_percent', [0,80,90,99.99,100],
                ['<80%','80-90%','90-99%','100%'],
                axes[4], 'Never Delinquent % vs Default', 'Never Delinquent %')

# Term
pct_term = pd.crosstab(closed['term'], closed['loan_status'], normalize='index') * 100
cnt_term = closed['term'].value_counts().sort_index()
pct_term.plot(kind='bar', stacked=True, ax=axes[5], legend=False)
for i, cat in enumerate(pct_term.index):
    axes[5].text(i, 101, f"n={cnt_term[cat]}", ha='center', fontsize=8, fontweight='bold')
axes[5].set_title('Term vs Default', fontsize=11)
axes[5].set_xlabel('Term (months)'); axes[5].set_ylabel('Percentage (%)')
axes[5].set_ylim(0, 112); axes[5].tick_params(axis='x', rotation=0)

# Add shared legend
handles, lbls = axes[0].get_legend_handles_labels()
fig.legend(handles, ['Charged Off','Fully Paid'], loc='upper right',
           title='Loan Status', fontsize=10)
plt.suptitle('Bivariate Analysis — Key Features vs Loan Status', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## Section 5 — Multivariate Analysis

> **Goal:** Test whether combinations of features reveal patterns not visible individually.
> All 5 hypotheses tested. Results interpreted with sample size in mind.

In [ ]:
# ── HYPOTHESIS 1: Annual Income + Loan Amount + Loan Status ──
# Does loan amount impact default differently across income levels?

h1 = closed.copy()
bins_inc = [0,25000,50000,75000,100000,150000,float('inf')]
bins_loan = [0,5000,10000,20000,30000,40000]
h1['income_group'] = pd.cut(h1['annual_income'], bins=bins_inc,
    labels=['<25K','25K-50K','50K-75K','75K-100K','100K-150K','>150K'], include_lowest=True)
h1['loan_group'] = pd.cut(h1['loan_amount'], bins=bins_loan,
    labels=['<5K','5K-10K','10K-20K','20K-30K','30K-40K'], include_lowest=True)

print('=== H1: Income Group × Loan Amount → Default Rate ===')
ct = pd.crosstab([h1['income_group'], h1['loan_group']], h1['loan_status'], normalize='index').mul(100).round(2)
print(ct)
print()
ct_counts = pd.crosstab([h1['income_group'], h1['loan_group']], h1['loan_status'])
print('Sample sizes:')
print(ct_counts)

# INSIGHT: No consistent pattern found. Lower-income borrowers with larger loans
# did not show reliably higher default rates. Most sub-groups have 0 Charged Off
# cases — sample too small for meaningful conclusions.

In [ ]:
# ── HYPOTHESIS 2: DTI + Interest Rate + Loan Status ──────────
# Does high debt burden + high borrowing cost increase default risk?

print('=== H2: DTI Group × Interest Rate Group → Default Rate ===')
h2_pct = closed.groupby(['debt_to_income_group', 'interest_rate_group'])['loan_status'].value_counts(normalize=True).mul(100).round(2)
print(h2_pct)
print()
print('Sample sizes:')
print(pd.crosstab([closed['debt_to_income_group'], closed['interest_rate_group']], closed['loan_status']))

# INSIGHT: No consistent upward trend found. The largest sub-group (DTI 10-20% +
# interest 10-20%) has 0 defaults out of 110 loans. Very small counts in high-DTI
# + high-interest combinations make conclusions unreliable. Hypothesis not supported.

In [ ]:
# ── HYPOTHESIS 3: Emp Length + Term + Loan Status ────────────
# Does employment stability matter more for longer-term loans?

h3 = closed.copy()
emp_order = ['Unknown','<1 year','1 year','2 years','3 years','4 years',
             '5 years','6 years','7 years','8 years','9 years','10+ years']
h3['emp_length'] = h3['emp_length'].astype(str)

print('=== H3: Employment Length × Term → Default Rate ===')
ct_h3 = pd.crosstab([h3['emp_length'], h3['term']], h3['loan_status'],
                     normalize='index').mul(100).round(2)
print(ct_h3)
print()
print('Sample sizes:')
print(pd.crosstab([h3['emp_length'], h3['term']], h3['loan_status']))

# INSIGHT: Many sub-groups have 0-1 Charged Off observations. No reliable pattern
# found between employment stability and term. Sub-groups too small to conclude anything.

In [ ]:
# ── HYPOTHESIS 4: Grade + Loan Amount + Loan Status ──────────
# Do larger loans in riskier grades default more?

h4 = closed.copy()
h4['loan_group'] = pd.cut(h4['loan_amount'],
    bins=[0,5000,10000,20000,30000,40000],
    labels=['<5K','5K-10K','10K-20K','20K-30K','30K-40K'], include_lowest=True)

print('=== H4: Grade × Loan Amount → Default Rate ===')
ct_h4 = pd.crosstab([h4['grade'], h4['loan_group']], h4['loan_status'],
                     normalize='index').mul(100).round(2)
print(ct_h4)

# INSIGHT: Most grade × loan amount combinations have 0 Charged Off cases.
# The pattern is not reliable due to extremely small sample sizes per sub-group.

In [ ]:
# ── HYPOTHESIS 5: Annual Income + DTI + Loan Status ──────────
# Does lower income + higher DTI = higher default?

h5 = closed.copy()
h5['income_group'] = pd.cut(h5['annual_income'],
    bins=[0,25000,50000,75000,100000,150000,float('inf')],
    labels=['<25K','25K-50K','50K-75K','75K-100K','100K-150K','>150K'], include_lowest=True)
h5['dti_group'] = pd.cut(h5['debt_to_income'],
    bins=[0,10,20,30,40,float('inf')],
    labels=['<10%','10-20%','20-30%','30-40%','>40%'], include_lowest=True)

print('=== H5: Income Group × DTI Group → Default Rate ===')
ct_h5 = pd.crosstab([h5['income_group'], h5['dti_group']], h5['loan_status'],
                     normalize='index').mul(100).round(2)
print(ct_h5[ct_h5.get('Charged Off', pd.Series()).fillna(0) > 0] if 'Charged Off' in ct_h5.columns else ct_h5)
print()
print('Sample sizes:')
print(pd.crosstab([h5['income_group'], h5['dti_group']], h5['loan_status']).sum(axis=1).sort_values(ascending=False).head(10))

# INSIGHT: No consistent pattern found across income-DTI combinations.
# Sample too small — most sub-groups contain 0 Charged Off cases.

---
## Section 6 — Executive Summary

> Written for a stakeholder who will never open the notebook.
> Each bullet = one finding + one specific conclusion.

---

### Key Findings

*Findings are ordered by strength of the observed signal. All interpretations should be read in the context of the dataset limitations described in Finding 1.*

---

**1. The sample size is the primary constraint on every finding in this analysis.**
The closed loan pool contains 452 loans with only 7 Charged Off cases (445 Fully Paid) — a 1.55% default rate. With 7 defaults, a single additional case shifts any group's rate by over 1 percentage point. No multivariate pattern tested survived this constraint: most sub-group combinations contained zero Charged Off cases. A substantially larger number of default observations would be required before any of these signals could support a predictive model or a policy recommendation.

---

**2. DTI showed the clearest directional tendency among the variables analysed.**
Borrowers in the 30–40% DTI band had an observed default rate of 2.86%, compared to 0.55% for the 10–20% band — a roughly 5× difference. This was the only variable where a consistent directional pattern was visible across groups. However, the underlying counts are small (35 loans in the 30–40% band, 2 of which defaulted), so this observation should be treated as a hypothesis to test on a larger dataset, not as a basis for a lending policy change.

---

**3. Income verification status is associated with different observed default rates.**
Not Verified borrowers defaulted at 1.94% vs 1.09% for Source Verified — approximately a 2× difference. The dataset records only the outcome by verification type; it does not establish why this difference exists. Greater uncertainty associated with unverified applications is a plausible interpretation, but the cause cannot be inferred from this data alone.

---

**4. All other variables — grade, homeownership, income, interest rate, installment — showed no reliable pattern.**
Default rates across these features were inconsistent across groups, with no directional trend. The absence of a pattern is itself a finding: it suggests these variables, at least individually, are not strong standalone predictors of default in this sample.

---

**5. A predictive model cannot be reliably built from this data.**
A model trained on 7 positive cases would likely be unstable and unreliable — its predictions driven by noise rather than genuine signal. To explore risk modelling, a larger dataset with substantially more default cases is needed. The full Lending Club dataset (2.2M rows) or the Home Credit Default Risk dataset (307K rows) would be appropriate next steps.

---

### Limitations

| Limitation | Impact |
|---|---|
| Only 7 Charged Off cases in closed loans | All default rates are highly sensitive to individual observations |
| Bins chosen manually, not statistically | Different cutoffs could produce different-looking patterns |
| Cross-sectional data only | No information on borrower behaviour over time |
| Closed loans are 4.5% of the dataset | Fully Paid / Charged Off loans may not represent the full portfolio |
| No causal inference possible | Associations observed here do not establish what causes default |

---
*Full code and charts available in sections above.*